# Do pathology foundation models survive the smartphone? (PLISM-sm)
Kaggle, 2x T4, 12 h limit. Resumable: re-run all cells after a restart; finished models are skipped.
Prereqs: Internet ON; add the code folder as a dataset (or upload to /kaggle/working/plism-study); add the PLISM files as an input dataset; add Kaggle secret `HF_TOKEN` (with access granted to the gated models).

In [ ]:
!pip -q install timm transformers huggingface_hub scikit-learn scipy einops einops_exts   # einops_exts: CONCH v1.5 (TITAN)
!pip -q install git+https://github.com/mahmoodlab/CONCH.git   # only needed for the 'conch' model

In [ ]:
import glob, os, subprocess, sys, time
from pathlib import Path
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login; login(os.environ["HF_TOKEN"])
except Exception as e:
    print("no HF_TOKEN secret (gated models will fail):", e)

CODE = next(Path(p).parent for p in glob.glob("/kaggle/input/**/extract.py", recursive=True) + glob.glob("/kaggle/working/**/extract.py", recursive=True))
# PLISM-sm is streamed directly from Figshare (CC BY 4.0); nothing is extracted to disk
TAR = "https://ndownloader.figshare.com/files/41397708"
CSV = "/kaggle/working/PLISM_sm_en.csv"
if not Path(CSV).exists():
    import urllib.request; urllib.request.urlretrieve("https://ndownloader.figshare.com/files/41397675", CSV)
EMB, SMOKE, RES = "/kaggle/working/emb", "/kaggle/working/smoke", "/kaggle/working/results"
# open models first; gated ones need approved access on Hugging Face
MODELS = ["vitb16_in", "dinov2_l", "hoptimus0", "uni", "uni2h", "conch", "conchv15", "virchow2",
          "hoptimus1", "h0mini", "phikon2", "gigapath"]
BUDGET_H, T0 = 11.0, time.time()
print(CODE, TAR, CSV, sep="\n")

def run(args):
    return subprocess.run([sys.executable, str(CODE / "extract.py"), "--tar", TAR, "--csv", CSV, *args]).returncode == 0

Smoke test: first 256 tiles, every model (checks access + loading + shapes before spending hours).

In [ ]:
import numpy as np
ok = {}
for m in ["debug"] + MODELS:
    ok[m] = run(["--model", m, "--out", SMOKE, "--limit", "256", "--batch", "32"])
print({m: v for m, v in ok.items()})
for m in MODELS:
    f = Path(SMOKE) / f"{m}.npy"
    if f.exists(): print(m, np.load(f).shape)
MODELS = [m for m in MODELS if ok.get(m)]   # drop models that failed to load (see log above)
print("will run:", MODELS)
!head -3 /kaggle/working/smoke/index.csv

Full extraction (one streaming pass of the tar per model; resumable).

In [ ]:
for m in MODELS:
    if (time.time() - T0) / 3600 > BUDGET_H:
        print("time budget reached; re-run the notebook in a new session to continue"); break
    run(["--model", m, "--out", EMB, "--batch", "64"])
!ls -la /kaggle/working/emb

Optional image-level fix: Reinhard stain normalisation to the Hamamatsu S360 reference (one model is enough for a comparison).

In [ ]:
# run(["--model", "uni", "--out", EMB, "--stain-norm", "reinhard", "--stain-ref", "S360"])

Analysis (CPU only; cached per model, so it can also be re-run).

In [ ]:
!python {CODE}/analysis.py --emb {EMB} --results {RES} --boot 1000 --folds 5
!cd /kaggle/working && zip -qr results.zip results && ls -la results.zip